## Imports

In [ ]:
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
import spacy
! python -m spacy download fr_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 40.4 MB/s  0:00:00 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')


In [3]:
nlp = spacy.load("fr_core_news_sm", disable=["parser", "ner"])

## Loading the metadata file

In [ ]:
# Load the CSV data - metadata
df = pd.read_csv('../data/archelect_search.csv')

# Extract year and filter
df['year'] = pd.to_datetime(df['date'], errors='coerce').dt.year
df = df[df['contexte-election'] == "législatives"]

## Extracting texts and joining w/ metadata file

In [ ]:
# Extract texts from text files 
records = []

for root, dirs, files in os.walk('../text_files'):
    for file in files:
        if file.endswith('.txt') :
            file_id = file[:-4]

            if file_id in df['id'].values:
                file_path = os.path.join(root, file)
            
            try:
                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    content = f.read()
                    
                    records.append({
                        "id": file_id,
                        "text": content,
                        "file_name": file
                    })
                    
            except Exception as e:
                print(f"Error reading {file_path}: {e}")

df_texts = pd.DataFrame(records)

In [12]:
# Merge w/ metadata
df_final = df.merge(df_texts, on="id", how="inner")

print(f"Total number of documents : {len(df_final)}")

# Stats
lengths = df_final["text"].apply(lambda x: len(x.split()))
print(f"Average length: {lengths.mean():.0f}")
print(f"Min: {lengths.min()}")
print(f"Max: {lengths.max()}")

Total number of documents : 21167
Average length: 712
Min: 19
Max: 4002


## Lemmatization

In [ ]:
df_final['lemmatized_text'] = [" ".join([token.lemma_ for token in doc]) for doc in nlp.pipe(df_final['text'])]

In [ ]:
df_final.head()

## Save the complete dataframe 

In [14]:
df_final.to_pickle("../data/df_final.pkl")